# Phase 4 — NLP Feature Extraction
## Notebook 04.03 — Transformer Embeddings and Dimensionality Reduction

### Goal
Complete Phase 4 with frozen general BERT embeddings, frozen financial FinBERT embeddings, compatible FinBERT sentiment probabilities, and smoke-test dimensionality-reduction recipes.

### Scope
This notebook:

- runs both models on `text_title_description` and `Filtered_Text`;
- records model/tokenizer identifiers, requested and resolved revisions, max length, pooling, truncation, padding, device, and dtype;
- saves small smoke embeddings as float32 arrays with a shared `source_row_id` mapping;
- compares structural output quality, runtime, and memory;
- runs PCA on dense embedding smoke matrices;
- runs TruncatedSVD on sparse TF-IDF smoke matrices;
- compiles the final Phase 4 method benchmark;
- provides a resumable local pipeline for full frozen embedding extraction.

### Leakage boundary
Frozen external embeddings and FinBERT probabilities do not learn parameters from this dataset. However, TF-IDF, PCA, and TruncatedSVD learn corpus-level parameters. Their outputs in this notebook are diagnostic smoke tests only. Final TF-IDF/PCA/SVD estimators must be fit on the training split after Phase 6 defines the temporal split.

## 1. Imports and repository paths

### Goal
Load reusable Stage 1–3 modules without machine-specific absolute paths.

### Decision
The notebook remains educational, while reusable extraction, reduction, saving, and resume logic stays in `src`.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def locate_repository_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "1_data_acquisition").exists()
            and (candidate / "3_text_preprocessing").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Expected Phase 1 and Phase 3 folders."
    )


PROJECT_ROOT = locate_repository_root()
PHASE4_DIR = PROJECT_ROOT / "4_nlp_feature_extraction"
if str(PHASE4_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE4_DIR))

from src.paths import resolve_project_path
from src.reduction_features import (
    compile_phase4_method_benchmark,
    run_pca_smoke,
    run_svd_smoke,
    save_reduction_smoke_outputs,
    write_stage3_metadata,
)
from src.tfidf_features import load_feature_recipes, run_tfidf_smoke_benchmark
from src.transformer_features import (
    extract_transformer_to_memmap,
    load_transformer_recipes,
    make_transformer_smoke_sample,
    run_transformer_smoke_benchmark,
    save_transformer_smoke_outputs,
)

print("Project root:", PROJECT_ROOT)
print("Phase 4 directory:", PHASE4_DIR)
print("PyTorch:", torch.__version__)
print("Detected accelerator:", "CUDA" if torch.cuda.is_available() else "CPU/MPS check at runtime")

Project root: C:\Users\sepehr\PycharmProjects\FinancialNLP
Phase 4 directory: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction
PyTorch: 2.13.0+cpu
Detected accelerator: CPU/MPS check at runtime


### Result
The notebook resolves all paths from the repository and imports the same reusable code covered by tests.

### Interpretation
This avoids duplicating complex batching or resume logic inside notebook cells.

### Decision
Keep the notebook as the transparent execution and interpretation layer.

## 2. Load Stage 2 and Stage 3 recipes

### Goal
Inspect exact TF-IDF, transformer, and reduction settings before reading data or downloading models.

### Decision
Model recipes use a requested revision (`main`) and record the exact resolved Hub commit SHA at runtime. This keeps the configuration readable while preserving reproducibility in generated metadata.

In [2]:
TFIDF_CONFIG_PATH = PHASE4_DIR / "configs" / "feature_recipes.yaml"
TRANSFORMER_CONFIG_PATH = PHASE4_DIR / "configs" / "transformer_recipes.yaml"

tfidf_config = load_feature_recipes(TFIDF_CONFIG_PATH)
transformer_config = load_transformer_recipes(TRANSFORMER_CONFIG_PATH)

input_cfg = transformer_config["input"]
smoke_cfg = transformer_config["smoke_test"]
output_cfg = transformer_config["output"]

INPUT_PATH = resolve_project_path(PROJECT_ROOT, input_cfg["relative_path"])
TRANSFORMER_SMOKE_DIR = resolve_project_path(
    PROJECT_ROOT, output_cfg["transformer_smoke_directory_relative_path"]
)
REDUCTION_SMOKE_DIR = resolve_project_path(
    PROJECT_ROOT, output_cfg["reduction_smoke_directory_relative_path"]
)
TRANSFORMER_BENCHMARK_PATH = resolve_project_path(
    PROJECT_ROOT, output_cfg["transformer_benchmark_relative_path"]
)
TRANSFORMER_METADATA_PATH = resolve_project_path(
    PROJECT_ROOT, output_cfg["transformer_metadata_relative_path"]
)
REDUCTION_BENCHMARK_PATH = resolve_project_path(
    PROJECT_ROOT, output_cfg["reduction_benchmark_relative_path"]
)
STAGE3_METADATA_PATH = resolve_project_path(
    PROJECT_ROOT, output_cfg["metadata_relative_path"]
)
FINAL_BENCHMARK_PATH = resolve_project_path(
    PROJECT_ROOT, output_cfg["final_benchmark_relative_path"]
)
FULL_EMBEDDING_DIR = resolve_project_path(
    PROJECT_ROOT, transformer_config["full_run"]["output_directory_relative_path"]
)

model_recipe_table = pd.DataFrame(
    [
        {
            "name": name,
            "model_id": spec["model_id"],
            "requested_revision": spec["revision"],
            "tokenizer_id": spec["tokenizer_id"],
            "task": spec["task"],
            "max_length": spec["max_length"],
            "pooling": spec["pooling"],
            "truncation": spec["truncation"],
            "padding": spec["padding"],
            "output_dtype": spec["output_dtype"],
            "sentiment_probabilities": spec["sentiment_probabilities"],
        }
        for name, spec in transformer_config["models"].items()
    ]
)
display(model_recipe_table)
print("Transformer smoke sample size:", smoke_cfg["sample_size"])
print("Transformer smoke batch size:", smoke_cfg["batch_size"])
print("PCA components requested:", transformer_config["reduction"]["pca"]["n_components"])
print("SVD components requested:", transformer_config["reduction"]["truncated_svd"]["n_components"])

,name,model_id,requested_revision,tokenizer_id,task,max_length,pooling,truncation,padding,output_dtype,sentiment_probabilities
0,bert,google-bert/bert-base-uncased,main,google-bert/bert-base-uncased,embedding,256,masked_mean,True,longest,float32,False
1,finbert,ProsusAI/finbert,main,ProsusAI/finbert,sequence_classification,256,masked_mean,True,longest,float32,True


Transformer smoke sample size: 96
Transformer smoke batch size: 8
PCA components requested: 16
SVD components requested: 16


### Interpretation
Both models use a fair, shared input policy: masked-mean pooling, `max_length=256`, truncation enabled, dynamic batch padding, and float32 output.

General BERT is an English general-domain baseline. FinBERT shares BERT's architecture but was adapted to financial language and fine-tuned for three-way financial sentiment. Its configured output order is always normalized to `positive`, `neutral`, `negative`, even though the model's internal class order differs.

## 3. Read Phase 3 text and carry forward Stage 1 findings

### Goal
Load only the identifier and two representations, then explicitly retain null/empty behavior discovered in Stage 1.

### Decision
Empty rows are never deleted. They remain mapped to their original `source_row_id`. Unlike TF-IDF, a transformer can produce a non-zero embedding for an empty string because special tokens are still processed; therefore empty-input counts and zero-output counts are separate diagnostics.

In [3]:
required_columns = [input_cfg["id_column"], *input_cfg["representations"]]
if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Phase 3 Parquet was not found: {INPUT_PATH}. Run Phase 3 first."
    )

text_df = pd.read_parquet(INPUT_PATH, columns=required_columns)
print("Loaded shape:", text_df.shape)
print("ID missing values:", int(text_df[input_cfg["id_column"]].isna().sum()))
print("ID duplicate rows:", int(text_df[input_cfg["id_column"]].duplicated(keep=False).sum()))

coverage_rows = []
for representation in input_cfg["representations"]:
    normalized = text_df[representation].astype("string").fillna("").str.strip()
    coverage_rows.append(
        {
            "representation": representation,
            "null_rows": int(text_df[representation].isna().sum()),
            "empty_or_null_rows": int(normalized.eq("").sum()),
            "nonempty_rows": int(normalized.ne("").sum()),
        }
    )
coverage = pd.DataFrame(coverage_rows)
display(coverage)

stage1_comparison_path = PHASE4_DIR / "data" / "reports" / "representation_comparison.csv"
if stage1_comparison_path.is_file():
    print("Stage 1 representation comparison:")
    display(pd.read_csv(stage1_comparison_path))

Loaded shape: (88936, 3)
ID missing values: 0
ID duplicate rows: 0


,representation,null_rows,empty_or_null_rows,nonempty_rows
0,text_title_description,0,76,88860
1,Filtered_Text,1,1,88935


Stage 1 representation comparison:


,representation,row_count,null_count,null_pct,empty_count,empty_pct,nonempty_count,nonempty_pct,median_word_count,p95_word_count,max_word_count,median_character_count,p95_character_count,max_character_count,very_short_word_count,very_short_word_pct_nonempty,very_short_character_count,very_short_character_pct_nonempty
0,text_title_description,88936,0,0.0000,76,0.0855,88860,99.9145,31.0000,72.0000,958,199.0000,456.0000,5705,0,0.0000,0,0.0000
1,Filtered_Text,88936,1,0.0011,1,0.0011,88935,99.9989,60.0000,318.0000,1636,454.0000,"2,432.0000",12724,530,0.5959,468,0.5262


### Interpretation
The current approved Stage 1 run reported 76 empty `text_title_description` rows and one null/empty `Filtered_Text` row. `Filtered_Text` is also substantially longer, so it is expected to have a higher truncation rate and slower transformer processing under the same maximum length.

### Decision
Keep the same `max_length` for a controlled technical comparison and report truncation rather than silently changing model inputs between representations.

## 4. Build a fixed transformer smoke sample

### Goal
Create a sample that is reproducible, includes representative empty-row edge cases, and still contains mostly normal news text.

### Difference from Stage 2
Stage 2 included all empty rows in a 3,000-row TF-IDF sample. A 96-row transformer sample would become dominated by the 76 empty title/description rows. This sampler therefore keeps only a configured maximum number of empty rows per representation and fills the remainder with deterministic random rows.

In [4]:
transformer_sample = make_transformer_smoke_sample(
    text_df,
    id_column=input_cfg["id_column"],
    representations=input_cfg["representations"],
    sample_size=smoke_cfg["sample_size"],
    random_state=smoke_cfg["random_state"],
    max_empty_rows_per_representation=smoke_cfg["max_empty_rows_per_representation"],
)
print("Smoke sample shape:", transformer_sample.shape)
print("Source order preserved:", transformer_sample[input_cfg["id_column"]].is_monotonic_increasing)
for representation in input_cfg["representations"]:
    normalized = transformer_sample[representation].astype("string").fillna("").str.strip()
    print(representation, "empty rows in sample:", int(normalized.eq("").sum()))

display(transformer_sample.head())

Smoke sample shape: (96, 3)
Source order preserved: True
text_title_description empty rows in sample: 4
Filtered_Text empty rows in sample: 1


,source_row_id,text_title_description,Filtered_Text
0,5798,"Regulation News Still Moves Bitcoin Prices, BIS Report Says A new report from the Bank of International Settlements (BIS) contends that bitcoin markets are swayed by news event...",new report bank international settlements bis contends markets swayed news events related regulation
1,8516,Opinion: there is practically no chances for the approval of bitcoin -etf in 2019 CNBC entrepreneur and cryptocurrency expert Brian Kelly believes that in 2019 the expected exc...,cnbc entrepreneur cryptocurrency expert brian kelly believes expected exchange funds etf expected many also receive approval american regulators
2,9136,Japan’s FSA Reports a 36% Decrease in Cryptocurrency Related Enquiries in Q4 of 2018 The Japanese Financial Services Agency has noted a drop in the number of inquiries related ...,japanese financial services agency noted drop number inquiries related digital currencies regulator noted fourth quarter received inquiries cryptocurrencies compared inquiries ...
3,10443,"BlockFi Registers 10,000 Customers for Its Crypto Interest-Bearing Accounts According to a Bloomberg report published March 20 2019, a significant number of risk-averse cryptoc...",according bloomberg report published march significant number risk-averse cryptocurrency investors flocked blockfi fintech startup offers healthy interest rate percent per annu...
4,12600,"Amazon Patents PoW System, Some Optimists Expect ‘Amazon Coin’ to Appear 🦄The recent patent of e-commerce giant Amazon has been approved, now the company can legally build a Pr...",recent patent e-commerce giant amazon approved company legally build proof-of-work blockchain system similar


## 5. Run frozen BERT and FinBERT smoke extraction

### Goal
Generate four dense embedding matrices:

- BERT × `text_title_description`
- BERT × `Filtered_Text`
- FinBERT × `text_title_description`
- FinBERT × `Filtered_Text`

FinBERT additionally produces two probability matrices with columns ordered as `positive`, `neutral`, `negative`.

### Practical note
The first run downloads model/tokenizer files from Hugging Face. The two model repositories require substantial disk space. Set `RUN_TRANSFORMER_SMOKE=False` when reopening the notebook only to inspect previously saved outputs.

In [5]:
RUN_TRANSFORMER_SMOKE = True

if RUN_TRANSFORMER_SMOKE:
    transformer_result = run_transformer_smoke_benchmark(
        transformer_sample,
        id_column=input_cfg["id_column"],
        representations=input_cfg["representations"],
        models=transformer_config["models"],
        batch_size=smoke_cfg["batch_size"],
        device=smoke_cfg["device"],
        track_truncation=smoke_cfg["track_truncation"],
    )
    display(transformer_result["benchmark"])
    print(json.dumps(transformer_result["model_metadata"], indent=2))
else:
    print("Transformer smoke extraction skipped by configuration flag.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

C:\Users\sepehr\PycharmProjects\FinancialNLP\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sepehr\.cache\huggingface\hub\models--google-bert--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


,representation,model,model_id,resolved_revision,tokenizer_id,max_length,pooling,truncation,empty_input_rows,source_row_id_order_preserved,rows,dimensions,shape,dtype,elapsed_seconds,rows_per_second,approx_embedding_memory_mb,finite_values,zero_embedding_rows,mean_l2_norm,std_l2_norm,truncated_rows,truncated_row_pct,median_untruncated_tokens,max_untruncated_tokens,sentiment_probabilities,sentiment_probability_max_sum_error
0,text_title_description,bert,google-bert/bert-base-uncased,86b5e0934494bd15c9632b12f734a8a67f723594,google-bert/bert-base-uncased,256,masked_mean,True,4,True,96,768,"(96, 768)",float32,7.7870,12.3283,0.2812,True,0,8.7375,0.6659,0,0.0000,47.0000,118,False,NaN
1,Filtered_Text,bert,google-bert/bert-base-uncased,86b5e0934494bd15c9632b12f734a8a67f723594,google-bert/bert-base-uncased,256,masked_mean,True,1,True,96,768,"(96, 768)",float32,20.2727,4.7354,0.2812,True,0,8.6754,0.6481,17,17.7083,80.5000,717,False,NaN
2,text_title_description,finbert,ProsusAI/finbert,4556d13015211d73dccd3fdd39d39232506f3e43,ProsusAI/finbert,256,masked_mean,True,4,True,96,768,"(96, 768)",float32,8.1278,11.8114,0.2812,True,0,9.9664,1.0084,0,0.0000,47.0000,118,True,0.0000
3,Filtered_Text,finbert,ProsusAI/finbert,4556d13015211d73dccd3fdd39d39232506f3e43,ProsusAI/finbert,256,masked_mean,True,1,True,96,768,"(96, 768)",float32,21.0354,4.5637,0.2812,True,0,10.1123,0.8297,17,17.7083,80.5000,717,True,0.0000


{
  "bert": {
    "model_id": "google-bert/bert-base-uncased",
    "requested_model_revision": "main",
    "resolved_model_revision": "86b5e0934494bd15c9632b12f734a8a67f723594",
    "model_class": "BertModel",
    "tokenizer_id": "google-bert/bert-base-uncased",
    "requested_tokenizer_revision": "main",
    "resolved_tokenizer_revision": "86b5e0934494bd15c9632b12f734a8a67f723594",
    "tokenizer_class": "BertTokenizer",
    "hidden_size": 768,
    "task": "embedding",
    "max_length": 256,
    "pooling": "masked_mean",
    "truncation": true,
    "padding": "longest",
    "model_dtype": "float32",
    "output_dtype": "float32",
    "sentiment_probabilities": false,
    "sentiment_output_order": null,
    "device": "cpu",
    "torch_version": "2.13.0+cpu",
    "transformers_version": "5.14.1",
    "python_version": "3.12.7"
  },
  "finbert": {
    "model_id": "ProsusAI/finbert",
    "requested_model_revision": "main",
    "resolved_model_revision": "4556d13015211d73dccd3fdd39d3923250

### Structural quality checks
A successful result must satisfy all of the following:

- exactly one embedding row per `source_row_id`;
- float32 output;
- finite values only;
- the configured hidden dimension;
- unchanged row order;
- valid probability rows summing to one when sentiment output is enabled.

These checks establish artifact integrity, not predictive quality. Semantic quality must later be measured against Phase 5 labels using Phase 6 splits and Phase 7 evaluation.

## 6. Save transformer smoke artifacts

### Goal
Save small reproducible arrays and complete metadata without adding model weights or full-dataset embeddings to Git.

In [6]:
if RUN_TRANSFORMER_SMOKE:
    transformer_output_files = save_transformer_smoke_outputs(
        transformer_result,
        output_directory=TRANSFORMER_SMOKE_DIR,
        benchmark_path=TRANSFORMER_BENCHMARK_PATH,
        metadata_path=TRANSFORMER_METADATA_PATH,
        id_column=input_cfg["id_column"],
        config=transformer_config,
    )
    display(pd.DataFrame(transformer_output_files.items(), columns=["artifact", "path"]))

,artifact,path
0,source_row_ids,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\transformer_smoke_source_row_ids.csv
1,embeddings::text_title_description__bert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\text_title_description__bert_embeddings.npy
2,embeddings::Filtered_Text__bert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\Filtered_Text__bert_embeddings.npy
3,embeddings::text_title_description__finbert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\text_title_description__finbert_embeddings.npy
4,embeddings::Filtered_Text__finbert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\Filtered_Text__finbert_embeddings.npy
5,sentiment::text_title_description__finbert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\text_title_description__finbert_sentiment_probabilities.npy
6,sentiment::Filtered_Text__finbert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\Filtered_Text__finbert_sentiment_probabilities.npy
7,model_metadata,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\model_metadata.json
8,benchmark,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reports\transformer_smoke_benchmark.csv
9,metadata,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\transformer_smoke\transformer_smoke_metadata.json


## 7. Compare BERT and FinBERT

### Goal
Compare dimensions, elapsed time, throughput, approximate float32 memory, truncation, and structural integrity.

### Expected interpretation
Both selected models use BERT-base hidden size 768, so their raw embedding dimensions and float32 memory per row are normally equal. Runtime can still differ because FinBERT also computes a classification head and returns sentiment logits. Representation length is likely to dominate runtime and truncation differences.

In [7]:
if RUN_TRANSFORMER_SMOKE:
    comparison_columns = [
        "representation",
        "model",
        "model_id",
        "resolved_revision",
        "rows",
        "dimensions",
        "elapsed_seconds",
        "rows_per_second",
        "approx_embedding_memory_mb",
        "empty_input_rows",
        "truncated_rows",
        "truncated_row_pct",
        "mean_l2_norm",
        "std_l2_norm",
        "finite_values",
        "zero_embedding_rows",
        "sentiment_probabilities",
    ]
    display(
        transformer_result["benchmark"][comparison_columns]
        .sort_values(["representation", "model"])
        .reset_index(drop=True)
    )

,representation,model,model_id,resolved_revision,rows,dimensions,elapsed_seconds,rows_per_second,approx_embedding_memory_mb,empty_input_rows,truncated_rows,truncated_row_pct,mean_l2_norm,std_l2_norm,finite_values,zero_embedding_rows,sentiment_probabilities
0,Filtered_Text,bert,google-bert/bert-base-uncased,86b5e0934494bd15c9632b12f734a8a67f723594,96,768,20.2727,4.7354,0.2812,1,17,17.7083,8.6754,0.6481,True,0,False
1,Filtered_Text,finbert,ProsusAI/finbert,4556d13015211d73dccd3fdd39d39232506f3e43,96,768,21.0354,4.5637,0.2812,1,17,17.7083,10.1123,0.8297,True,0,True
2,text_title_description,bert,google-bert/bert-base-uncased,86b5e0934494bd15c9632b12f734a8a67f723594,96,768,7.7870,12.3283,0.2812,4,0,0.0000,8.7375,0.6659,True,0,False
3,text_title_description,finbert,ProsusAI/finbert,4556d13015211d73dccd3fdd39d39232506f3e43,96,768,8.1278,11.8114,0.2812,4,0,0.0000,9.9664,1.0084,True,0,True


### Decision
Do not declare BERT or FinBERT the final winner from runtime or vector norms. Carry both feature families into controlled Phase 7 experiments. FinBERT has a domain-specific prior and sentiment output; general BERT provides a useful baseline against which that specialization can be measured.

## 8. Inspect FinBERT sentiment probabilities

### Goal
Verify the canonical probability column order and summarize output behavior without treating it as a market-direction label.

### Important distinction
FinBERT predicts linguistic financial sentiment. It does not directly predict future Bitcoin price direction, return, or trading profitability. Those targets belong to Phase 5.

In [8]:
if RUN_TRANSFORMER_SMOKE:
    sentiment_rows = []
    for key, probabilities in transformer_result["sentiment_probabilities"].items():
        sentiment_rows.append(
            {
                "output": key,
                "rows": probabilities.shape[0],
                "positive_mean": float(probabilities[:, 0].mean()),
                "neutral_mean": float(probabilities[:, 1].mean()),
                "negative_mean": float(probabilities[:, 2].mean()),
                "max_probability_sum_error": float(
                    np.abs(probabilities.sum(axis=1) - 1.0).max()
                ),
            }
        )
    display(pd.DataFrame(sentiment_rows))

,output,rows,positive_mean,neutral_mean,negative_mean,max_probability_sum_error
0,text_title_description__finbert,96,0.2787,0.4737,0.2476,0.0000
1,Filtered_Text__finbert,96,0.2527,0.5868,0.1604,0.0000


## 9. PCA smoke test for dense embeddings

### Goal
Demonstrate a valid PCA recipe on the four dense embedding matrices.

### Leakage warning
This PCA fit is only a smoke test. The final PCA estimator must be fit on training embeddings only, then used unchanged to transform validation and test embeddings.

In [9]:
if RUN_TRANSFORMER_SMOKE:
    pca_result = run_pca_smoke(
        transformer_result["embeddings"],
        source_row_ids=transformer_result["source_row_ids"],
        settings=transformer_config["reduction"]["pca"],
    )
    display(pca_result["benchmark"])

,input_key,method,rows,input_dimensions,output_dimensions,shape,dtype,elapsed_seconds,approx_output_memory_mb,explained_variance_ratio_sum,finite_values,source_row_id_order_preserved,fit_scope
0,text_title_description__bert,pca,96,768,16,"(96, 16)",float32,0.0620,0.0059,0.6731,True,True,smoke_sample_only
1,Filtered_Text__bert,pca,96,768,16,"(96, 16)",float32,0.0090,0.0059,0.7102,True,True,smoke_sample_only
2,text_title_description__finbert,pca,96,768,16,"(96, 16)",float32,0.0086,0.0059,0.7373,True,True,smoke_sample_only
3,Filtered_Text__finbert,pca,96,768,16,"(96, 16)",float32,0.0086,0.0059,0.7479,True,True,smoke_sample_only


## 10. TruncatedSVD smoke test for sparse TF-IDF

### Goal
Apply TruncatedSVD directly to CSR TF-IDF matrices without converting the sparse inputs to dense arrays.

### Why not PCA for TF-IDF?
PCA centers input features and is naturally used here for dense transformer embeddings. TruncatedSVD does not require centering and can operate efficiently on sparse TF-IDF matrices. Its transformed output is dense, so the reduced arrays are stored as float32 NumPy files.

### Sample alignment
Stage 2 used a separate 3,000-row smoke sample. To keep PCA, SVD, and transformer rows directly comparable in this notebook, the Stage 2 TF-IDF recipes are rerun on the same 96-row Stage 3 sample. These vectorizers remain smoke-only.

In [10]:
# Tiny smoke samples can require lower document-frequency thresholds than the
# approved 3,000-row Stage 2 benchmark. Copy the recipe before changing it.
from copy import deepcopy

stage3_tfidf_recipes = deepcopy(tfidf_config["recipes"])
for recipe_name in ("word_tfidf", "character_tfidf"):
    stage3_tfidf_recipes[recipe_name]["vectorizer"]["min_df"] = 1
    stage3_tfidf_recipes[recipe_name]["vectorizer"]["max_df"] = 1.0

tfidf_stage3_result = run_tfidf_smoke_benchmark(
    transformer_sample,
    id_column=input_cfg["id_column"],
    representations=input_cfg["representations"],
    recipes=stage3_tfidf_recipes,
)

svd_result = run_svd_smoke(
    tfidf_stage3_result["matrices"],
    source_row_ids=tfidf_stage3_result["source_row_ids"],
    settings=transformer_config["reduction"]["truncated_svd"],
    allowed_recipe_names=transformer_config["reduction"]["truncated_svd"]["tfidf_recipes"],
)
display(svd_result["benchmark"])

,input_key,method,rows,input_dimensions,output_dimensions,shape,dtype,elapsed_seconds,approx_output_memory_mb,explained_variance_ratio_sum,finite_values,source_row_id_order_preserved,fit_scope
0,text_title_description__word_tfidf,truncated_svd,96,4058,16,"(96, 16)",float32,0.0204,0.0059,0.1974,True,True,smoke_sample_only
1,text_title_description__character_tfidf,truncated_svd,96,6000,16,"(96, 16)",float32,0.0329,0.0059,0.2683,True,True,smoke_sample_only
2,text_title_description__word_character_tfidf,truncated_svd,96,10058,16,"(96, 16)",float32,0.0906,0.0059,0.2285,True,True,smoke_sample_only
3,Filtered_Text__word_tfidf,truncated_svd,96,6000,16,"(96, 16)",float32,0.0271,0.0059,0.2401,True,True,smoke_sample_only
4,Filtered_Text__character_tfidf,truncated_svd,96,6000,16,"(96, 16)",float32,0.0426,0.0059,0.3341,True,True,smoke_sample_only
5,Filtered_Text__word_character_tfidf,truncated_svd,96,12000,16,"(96, 16)",float32,0.0989,0.0059,0.2792,True,True,smoke_sample_only


## 11. Save reduction artifacts

### Goal
Save PCA/SVD smoke arrays, fitted smoke estimators, separate ID mappings, and a shared reduction benchmark.

### Decision
The estimator files are diagnostic and must never be promoted as final model preprocessors.

In [11]:
if RUN_TRANSFORMER_SMOKE:
    reduction_output_files = save_reduction_smoke_outputs(
        pca_result=pca_result,
        svd_result=svd_result,
        output_directory=REDUCTION_SMOKE_DIR,
        benchmark_path=REDUCTION_BENCHMARK_PATH,
        id_column=input_cfg["id_column"],
    )
    display(pd.DataFrame(reduction_output_files.items(), columns=["artifact", "path"]))

,artifact,path
0,pca_source_row_ids,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\pca_smoke_source_row_ids.csv
1,pca_array::text_title_description__bert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\text_title_description__bert_pca.npy
2,pca_estimator::text_title_description__bert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\text_title_description__bert_pca.joblib
3,pca_array::Filtered_Text__bert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\Filtered_Text__bert_pca.npy
4,pca_estimator::Filtered_Text__bert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\Filtered_Text__bert_pca.joblib
5,pca_array::text_title_description__finbert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\text_title_description__finbert_pca.npy
6,pca_estimator::text_title_description__finbert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\text_title_description__finbert_pca.joblib
7,pca_array::Filtered_Text__finbert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\Filtered_Text__finbert_pca.npy
8,pca_estimator::Filtered_Text__finbert,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\Filtered_Text__finbert_pca.joblib
9,svd_source_row_ids,C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reduction_smoke\svd_smoke_source_row_ids.csv


## 12. Compile the final Phase 4 benchmark

### Goal
Create one inventory covering:

- Stage 1 explainable row-level features;
- Stage 2 word, character, and combined TF-IDF;
- Stage 3 BERT and FinBERT embeddings;
- Stage 3 PCA and TruncatedSVD smoke reductions.

### Comparison limitation
Elapsed time and memory reflect different smoke sizes and the local hardware used for each notebook. They are engineering diagnostics, not a fair predictive leaderboard. Predictive quality remains explicitly `not_evaluated_in_phase4`.

In [12]:
if RUN_TRANSFORMER_SMOKE:
    stage1_metadata_path = PHASE4_DIR / "data" / "reports" / "stage1_run_metadata.json"
    stage2_benchmark_path = PHASE4_DIR / "data" / "reports" / "tfidf_smoke_benchmark.csv"
    reduction_benchmark = pd.concat(
        [pca_result["benchmark"], svd_result["benchmark"]], ignore_index=True
    )
    final_benchmark = compile_phase4_method_benchmark(
        stage1_metadata_path=stage1_metadata_path,
        stage2_benchmark_path=stage2_benchmark_path,
        transformer_benchmark=transformer_result["benchmark"],
        reduction_benchmark=reduction_benchmark,
    )
    FINAL_BENCHMARK_PATH.parent.mkdir(parents=True, exist_ok=True)
    final_benchmark.to_csv(FINAL_BENCHMARK_PATH, index=False)
    write_stage3_metadata(
        STAGE3_METADATA_PATH,
        transformer_outputs=transformer_output_files,
        reduction_outputs=reduction_output_files,
        final_benchmark_path=FINAL_BENCHMARK_PATH,
    )
    display(final_benchmark)
    print("Final benchmark:", FINAL_BENCHMARK_PATH)

,stage,method_family,representation,method,rows,dimensions,storage,elapsed_seconds,memory_mb,finite_values,fit_scope,semantic_quality_status
0,04_01,row_level_manual,both representations,explainable_row_features,88936,48,dense_parquet,182.6321,NaN,True,row_local_full_dataset,not_evaluated_in_phase4
1,04_02,tfidf,text_title_description,word_tfidf,3000,6000,csr_sparse_npz,0.5369,0.7334,True,smoke_sample_only,not_evaluated_in_phase4
2,04_02,tfidf,text_title_description,character_tfidf,3000,6000,csr_sparse_npz,1.2084,5.9007,True,smoke_sample_only,not_evaluated_in_phase4
3,04_02,tfidf,text_title_description,word_character_tfidf,3000,12000,csr_sparse_npz,1.6571,6.6226,True,smoke_sample_only,not_evaluated_in_phase4
4,04_02,tfidf,Filtered_Text,word_tfidf,3000,6000,csr_sparse_npz,1.5667,2.0778,True,smoke_sample_only,not_evaluated_in_phase4
5,04_02,tfidf,Filtered_Text,character_tfidf,3000,6000,csr_sparse_npz,4.3992,19.1135,True,smoke_sample_only,not_evaluated_in_phase4
6,04_02,tfidf,Filtered_Text,word_character_tfidf,3000,12000,csr_sparse_npz,5.9577,21.1798,True,smoke_sample_only,not_evaluated_in_phase4
7,04_03,frozen_transformer,text_title_description,bert,96,768,dense_float32_npy,7.7870,0.2812,True,frozen_external_model,not_evaluated_in_phase4
8,04_03,frozen_transformer,Filtered_Text,bert,96,768,dense_float32_npy,20.2727,0.2812,True,frozen_external_model,not_evaluated_in_phase4
9,04_03,frozen_transformer,text_title_description,finbert,96,768,dense_float32_npy,8.1278,0.2812,True,frozen_external_model,not_evaluated_in_phase4


Final benchmark: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reports\phase4_method_benchmark.csv


## 13. Optional full frozen embedding extraction

### Goal
Provide a complete local pipeline for all current rows without forcing a full run inside this notebook.

### Batch and resume behavior
Each model/representation job writes:

- a float32 `.npy` embedding memmap;
- an exact `source_row_id` CSV mapping;
- optional float32 FinBERT probability memmap;
- progress JSON after every batch;
- final metadata containing the exact resolved model revision and preprocessing contract.

Rerunning the same job with resume enabled continues from `next_index`. A fingerprint prevents accidental resume against a different dataset or recipe.

In [13]:
RUN_FULL_EMBEDDINGS = False

if RUN_FULL_EMBEDDINGS:
    full_cfg = transformer_config["full_run"]
    full_run_summaries = []
    for model_name, spec in transformer_config["models"].items():
        for representation in input_cfg["representations"]:
            summary = extract_transformer_to_memmap(
                text_df[[input_cfg["id_column"], representation]],
                id_column=input_cfg["id_column"],
                representation=representation,
                model_name=model_name,
                spec=spec,
                output_directory=FULL_EMBEDDING_DIR,
                batch_size=full_cfg["batch_size"],
                device=full_cfg["device"],
                resume=full_cfg["resume"],
                track_truncation=full_cfg["track_truncation"],
            )
            full_run_summaries.append(summary)
    display(pd.DataFrame(full_run_summaries))
else:
    print("Full extraction is disabled. Use the local CLI commands documented below.")

Full extraction is disabled. Use the local CLI commands documented below.


### Recommended local commands

Run one resumable job at a time from the Phase 4 folder:

```bash
cd 4_nlp_feature_extraction

python -m src.transformer_features \
  --config configs/transformer_recipes.yaml \
  --model bert \
  --representation text_title_description

python -m src.transformer_features \
  --config configs/transformer_recipes.yaml \
  --model bert \
  --representation Filtered_Text

python -m src.transformer_features \
  --config configs/transformer_recipes.yaml \
  --model finbert \
  --representation text_title_description

python -m src.transformer_features \
  --config configs/transformer_recipes.yaml \
  --model finbert \
  --representation Filtered_Text
```

On GPU, increase `full_run.batch_size` gradually while monitoring memory. On CPU, smaller batches reduce peak memory but will not remove the substantial compute cost of processing every row four times.

## 14. Phase 4 completion and handoff

### Completed Phase 4 outputs

1. **Row-local explainable features** — full-dataset Parquet keyed by `source_row_id`.
2. **TF-IDF recipes and sparse smoke matrices** — word, character, and combined for both representations.
3. **Frozen transformer pipeline** — BERT and FinBERT for both representations, float32 output, batching, mapping, and resume.
4. **FinBERT sentiment probabilities** — canonical `positive`, `neutral`, `negative` ordering when the model label contract is compatible.
5. **Reduction recipes** — PCA for dense embeddings and TruncatedSVD for sparse TF-IDF, smoke-only until split.
6. **Final method benchmark** — artifact dimensions, storage type, runtime, memory, and structural validation.

### Handoff to Phase 5 — market alignment and labeling

Phase 5 should create event-time market targets and retain `source_row_id`. It must not use publisher sentiment or FinBERT probabilities as ground-truth labels. Frozen text outputs and future market targets remain separate tables joined by ID only when needed.

### Handoff to Phase 6 — splitting and leakage control

Phase 6 owns the temporal train/validation/test split and should save split membership by `source_row_id`. TF-IDF vectorizers, PCA, TruncatedSVD, scalers, feature selection, and any learned preprocessing must be fit on training rows only.

### Handoff to Phase 7 — model selection and training

Phase 7 can compare approved feature families:

- Stage 1 manual row features;
- train-fitted TF-IDF;
- frozen BERT embeddings;
- frozen FinBERT embeddings;
- FinBERT sentiment probabilities as input features, not labels;
- train-fitted PCA/SVD reductions.

Every experiment should record representation, exact model revision, pooling, max length, truncation, feature recipe, split version, and target definition.